# 螺旋矩阵 II
## 问题
给你一个正整数 n，生成一个包含 1 到 n² 所有整数的 n × n 矩阵，要求元素按顺时针螺旋顺序排列

[力扣题目链接](https://leetcode.cn/problems/spiral-matrix-ii/)

示例 1:
- 输入：n = 3
- 输出：[[1,2,3],[8,9,4],[7,6,5]]

对应的矩阵为：

```text
1  2  3
8  9  4
7  6  5
```

示例 2:
- 输入：n = 1
- 输出：[[1]]

## 思路
本题可以直接模拟填数的过程：从左上角开始，按从左到右、从上到下、从右到左、从下到上的顺序填完一圈，再向内填下一圈

关键是每条边的范围要统一，否则四个角容易被重复填写，或者出现遗漏

和二分查找一样，这里也需要坚持循环不变量：**每条边都包含沿遍历方向的起点，不包含终点**。这样每个角都交给下一条边处理，每个位置只会被填写一次

## 解法
### 按圈模拟


用 start 和 end 表示当前这一圈的边界下标：
- start 初始为 0，表示这一圈的上边界和左边界
- end 初始为 n - 1，表示这一圈的下边界和右边界
- 用 value 保存下一个要填入的数字，初始为 1

矩阵下标使用 result[行][列]。每一圈按以下顺序填写：

| 遍历方向 | 固定下标 | 移动下标及范围 | 包含的角 | 不包含的角 |
| --- | --- | --- | --- | --- |
| 从左到右 | 行为 start | col：range(start, end) | 左上角 | 右上角 |
| 从上到下 | 列为 end | row：range(start, end) | 右上角 | 右下角 |
| 从右到左 | 行为 end | col：range(end, start, -1) | 右下角 | 左下角 |
| 从下到上 | 列为 start | row：range(end, start, -1) | 左下角 | 左上角 |

Python 的 range 不包含终止值：range(start, end) 依次得到 start 到 end - 1；range(end, start, -1) 依次得到 end 到 start + 1

因此，正向和反向遍历都遵守“包含起点，不包含终点”的规则

每填一个位置，就让 value 加 1。四条边填完后，让 start 加 1、end 减 1，进入内层

循环条件使用 start < end，只有边长至少为 2 时，才有完整的一圈：
- n 为偶数时，填完 n // 2 圈后，start > end，所有元素都已填完
- n 为奇数时，填完 n // 2 圈后，start == end，还剩一个中心位置，需要单独填写

以 n = 3 为例，第一圈的 start = 0、end = 2：

| 遍历方向 | 填写的位置（行，列） | 填入的数字 |
| --- | --- | --- |
| 从左到右 | (0,0)、(0,1) | 1、2 |
| 从上到下 | (0,2)、(1,2) | 3、4 |
| 从右到左 | (2,2)、(2,1) | 5、6 |
| 从下到上 | (2,0)、(1,0) | 7、8 |

第一圈填完后，start = end = 1，循环结束，再将 9 填入中心位置 (1,1)


In [ ]:
class solution:
    def generateMatrix(self, n: int) -> list[list[int]]:
        result = [[0] * n for _ in range(n)]
        start, end = 0, n - 1
        value = 1

        while start < end:
            # 上边：从左到右，不包含右上角
            for col in range(start, end):
                result[start][col] = value
                value += 1

            # 右边：从上到下，不包含右下角
            for row in range(start, end):
                result[row][end] = value
                value += 1

            # 下边：从右到左，不包含左下角
            for col in range(end, start, -1):
                result[end][col] = value
                value += 1

            # 左边：从下到上，不包含左上角
            for row in range(end, start, -1):
                result[row][start] = value
                value += 1

            start += 1
            end -= 1

        # 奇数阶矩阵还剩一个中心元素
        if n % 2 == 1:
            result[start][start] = value

        return result


时间复杂度：$O(n^2)$ 矩阵共有 n² 个位置，每个位置只填写一次；初始化矩阵也需要 $O(n^2)$ 时间

空间复杂度：$O(n^2)$ 需要保存 n × n 的结果矩阵；如果不计返回结果占用的空间，额外空间为 $O(1)$


### 易错点
- result[row][col] 的第一个下标是行，第二个下标是列；填写上下两条边时固定行，填写左右两条边时固定列
- 四条边都要遵守“包含起点，不包含终点”的规则，不能让相邻两条边都填写同一个角
- 反向遍历的步长为 -1，range(end, start, -1) 不包含 start
- 必须填完四条边后再更新 start 和 end，保证同一圈使用相同的边界
- 奇数阶矩阵需要单独处理中心元素；n = 1 时不会进入循环，直接填写中心位置即可
- 初始化二维列表时使用 [[0] * n for _ in range(n)]，不要使用 [[0] * n] * n：后者会让多行引用同一个列表，修改一行时其他行也会跟着变化
